# 05 · Exploration (EDA)

**Phase skill: describing a dataset before modeling it** — distributions, relationships,
group summaries, and a correlation matrix you will meet again in notebook 06.

The setup cell loads the same per-monster feature table the real pipeline uses
(`load_sd_features` from `src/analysis.py`): one row per monster from `sd_monsters`,
plus its best single-attack columns derived from `sd_attacks`.

After every plot there is a one-line writing prompt: **state what the chart claims.**
If you can't say it in a sentence, the chart isn't done.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))
from analysis import load_sd_features

DB_PATH = (Path.cwd().parent / "data" / "monsterlab.db").resolve()
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
df = load_sd_features(conn)
print(f"{len(df)} monsters x {df.shape[1]} columns")
df[["name", "level", "ac", "hp", "best_attack_bonus", "best_avg_damage",
    "best_num_attacks", "best_stat_mod", "best_round_damage"]].head()

## Exercise 5.1 — the LV distribution

Plot a histogram of `df["level"]` (one bin per level works well), then quantify its
asymmetry.

**Produce:** the plot, and `lv_skew` — the skewness of the level column as a float.

<details><summary>Hint 1 (concept)</summary>

Skewness is signed: it tells you which tail is long, not just that the distribution is lopsided.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `scipy.stats.skew` and `matplotlib`'s `ax.hist`.

</details>

In [ ]:
# Produce: a histogram of level, and lv_skew (float)

lv_skew = ...

print(f"skew = {lv_skew:.3f}")

In [ ]:
assert abs(lv_skew - 1.669) < 0.02, (
    f"lv_skew = {lv_skew:.3f}, expected about 1.67 -- check the column and the function's input")
print(f"PASS: skew = {lv_skew:.2f}, a strongly right-tailed distribution.")
print("Context: most monsters live at LV 0-7; the long right tail is a handful of bosses, so means will flatter the tail.")

**State in one sentence what this chart claims.**

*…*

## Exercise 5.2 — AC against LV

Scatter `ac` (y) against `level` (x), then measure the linear association.

**Produce:** the scatter, and `ac_lv_corr` — the Pearson correlation between `ac` and
`level` as a float.

<details><summary>Hint 1 (concept)</summary>

A correlation is a one-number summary of a scatter you should already have looked at -- the plot tells you whether the number is honest.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `DataFrame.corr` or `Series.corr`, and `ax.scatter`.

</details>

In [ ]:
# Produce: a scatter of ac vs level, and ac_lv_corr (float)

ac_lv_corr = ...

print(f"r = {ac_lv_corr:.3f}")

In [ ]:
assert abs(ac_lv_corr - 0.656) < 0.005, (
    f"ac_lv_corr = {ac_lv_corr:.3f}, expected about 0.656 -- Pearson, ac against level")
print(f"PASS: r = {ac_lv_corr:.2f}.")
print("Context: moderate at best -- AC climbs with LV but nowhere near tightly enough to predict it alone.")

**State in one sentence what this chart claims.**

*…*

## Exercise 5.3 — damage per round by LV band

Cut `level` into four bands — bins `[-1, 3, 7, 11, 30]` with labels
`["0-3", "4-7", "8-11", "12+"]` — and average `best_round_damage` inside each band.
Plot the band means as a bar chart.

**Produce:** `dmg_by_band` — a Series of the four band means, indexed by the band
labels, in the order above.

<details><summary>Hint 1 (concept)</summary>

Binning turns a continuous column into an ordered categorical one; after that it's an ordinary group-and-aggregate.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `pd.cut` and `groupby(...).mean` (pass `observed=True` to silence the categorical warning).

</details>

In [ ]:
# Produce: dmg_by_band (Series: mean best_round_damage per band) and its bar chart

dmg_by_band = ...

dmg_by_band

In [ ]:
assert len(dmg_by_band) == 4, f"dmg_by_band has {len(dmg_by_band)} entries, expected 4 bands"
expected_means = {"0-3": 4.54, "4-7": 10.30, "8-11": 20.70, "12+": 37.64}
for band, want in expected_means.items():
    assert band in dmg_by_band.index, f"band {band!r} is missing from the index"
    got = float(dmg_by_band[band])
    assert abs(got - want) < 0.05, f"mean for band {band} = {got:.2f}, expected {want:.2f}"
print("PASS: band means 4.5 / 10.3 / 20.7 / 37.6.")
print("Context: expected damage roughly doubles per band -- the tiers are steeper than the LV numbers look.")

**State in one sentence what this chart claims.**

*…*

## Exercise 5.4 — the correlation matrix

Compute the correlation matrix of the model's candidate columns, in this order:
`level`, `ac`, `hp`, `best_attack_bonus`, `best_avg_damage`, `best_num_attacks`,
`best_stat_mod`. Render it however you like (a plain styled dataframe is fine; so is
`ax.imshow`).

**Produce:** `corr_matrix` — the 7×7 correlation DataFrame.

<details><summary>Hint 1 (concept)</summary>

A correlation matrix is the pairwise version of what you did in 5.2 -- every cell is one scatter you didn't have to draw.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `DataFrame.corr`.

</details>

In [ ]:
# Produce: corr_matrix (7x7 DataFrame)

corr_matrix = ...

corr_matrix.round(2)

In [ ]:
assert corr_matrix.shape == (7, 7), f"corr_matrix is {corr_matrix.shape}, expected (7, 7)"
lv = corr_matrix["level"]
assert abs(lv["hp"] - 0.998) < 0.005, f"corr(level, hp) = {lv['hp']:.3f}, expected about 0.998"
assert abs(lv["best_attack_bonus"] - 0.892) < 0.005, (
    f"corr(level, best_attack_bonus) = {lv['best_attack_bonus']:.3f}, expected about 0.892")
assert abs(lv["best_num_attacks"] - 0.683) < 0.005, (
    f"corr(level, best_num_attacks) = {lv['best_num_attacks']:.3f}, expected about 0.683")
print("PASS: correlation matrix matches.")
print("Context: one of these numbers is not like the others. Sit with corr(level, hp) = 0.998 for a moment "
      "-- notebook 06 is about exactly that cell.")

**Two sentences this time:** which correlation is suspiciously strong, and what is your
best guess at *why* before notebook 06 tells you?

*…*